# TechTrack use case 

# Task 2: Sampling strategy

This notebook proposes and implements a class-aware, small-object-aware, and density-penalized sampling strategy for the TechTrack logistics dataset.

It will:
1. Scan the dataset (`.jpg/.png` + `.txt`) and compute per-image metadata.
2. Compute per-class image frequencies.
3. Assign to each image a primary class (the rarest class present in that image).
4. Compute a sampling weight per image:
   $w = w_{class} \times w_{small} \times w_{density}$
5. Convert weights into sampling probabilities and draw samples.
6. Export sampled subsets to CSV

In [1]:

import os
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import shutil

try:
    from tqdm import tqdm
except Exception:
    tqdm = lambda x, **k: x

import cv2

import sys
from pathlib import Path


## Dataset inspection

Before proposing the sampling strategy, we need to inspect our logistics dataset to detect class imbalance. This script computes dataset summary stats (how many images are empty, how many contain multiple objects, and how often small vs. large boxes appear based on normalized area `w*h` thresholds), plus two complementary class-imbalance views: image-level presence (how many images contain each class at least once) and box-level frequency (how many total boxes of each class).


In [4]:
# TechTrack logistics dataset scan:
# - class imbalance (image-level + box-level)
# - multi-object frames (multiple lines per .txt)
# - difficulty imbalance (small/medium/large boxes using YOLO normalized w*h)

from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent
DATASET_DIR = REPO_ROOT / "techtrack" / "storage" / "logistics"
NUM_CLASSES = 20

# Difficulty thresholds based on YOLO normalized area = w*h
SMALL_AREA_TH = 0.01     
LARGE_AREA_TH = 0.10     


SOURCE_SPLIT_TOKEN = "_" 

# HELPERS
def parse_yolo_txt(txt_path: Path):
    """
    Reads YOLO txt annotations.
    Each line: class_id x_center y_center w h
    Returns list of tuples: (class_id, x, y, w, h)
    """
    if not txt_path.exists():
        return []
    text = txt_path.read_text().strip()
    if not text:
        return []
    rows = []
    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            rows.append((c, x, y, w, h))
        except ValueError:
            # skip malformed lines
            continue
    return rows

# SCAN DATASET
# Pair images to txt: assumes same stem .jpg/.png -> .txt
image_paths = sorted(list(DATASET_DIR.glob("*.jpg")) + list(DATASET_DIR.glob("*.png")))
if not image_paths:
    raise FileNotFoundError(f"No .jpg/.png found in {DATASET_DIR.resolve()}")

records = []
box_class_counts = Counter()
image_class_presence = Counter()
source_counts = Counter()
small_box_images = 0
large_box_images = 0
multi_object_images = 0
empty_images = 0

# For distribution plots
all_areas = []

for img_path in image_paths:
    txt_path = img_path.with_suffix(".txt")
    ann = parse_yolo_txt(txt_path)

    source_id = infer_source_id(img_path)
    source_counts[source_id] += 1

    num_objects = len(ann)
    if num_objects == 0:
        empty_images += 1
        classes_in_img = set()
        has_small = False
        has_large = False
    else:
        if num_objects >= 2:
            multi_object_images += 1

        classes_in_img = {c for c, *_ in ann}
        for c, x, y, w, h in ann:
            box_class_counts[c] += 1
            area = max(0.0, w) * max(0.0, h)
            all_areas.append(area)

        areas = [max(0.0, w) * max(0.0, h) for c, x, y, w, h in ann]
        has_small = any(a < SMALL_AREA_TH for a in areas)
        has_large = any(a > LARGE_AREA_TH for a in areas)

        if has_small:
            small_box_images += 1
        if has_large:
            large_box_images += 1

        # Image-level class presence counts (each class counted once per image)
        for c in classes_in_img:
            image_class_presence[c] += 1

    records.append(
        dict(
            image=str(img_path),
            annotation=str(txt_path),
            source_id=source_id,
            num_objects=num_objects,
            is_empty=(num_objects == 0),
            num_unique_classes=len(classes_in_img),
            classes_sorted=sorted(list(classes_in_img)),
            has_small_object=has_small,
            has_large_object=has_large,
        )
    )

df = pd.DataFrame(records)

# SUMMARY STATS
n_images = len(df)
n_nonempty = n_images - empty_images

summary = {
    "images_total": n_images,
    "images_with_objects": n_nonempty,
    "images_empty": empty_images,
    "pct_empty": empty_images / n_images,
    "images_multi_object(>=2)": multi_object_images,
    "pct_multi_object_over_all": multi_object_images / n_images,
    "pct_multi_object_over_nonempty": (multi_object_images / n_nonempty) if n_nonempty else 0.0,
    f"images_with_small_object(area<{SMALL_AREA_TH})": small_box_images,
    f"pct_small_object_images": small_box_images / n_images,
    f"images_with_large_object(area>{LARGE_AREA_TH})": large_box_images,
    f"pct_large_object_images": large_box_images / n_images,
    "unique_sources(proxy)": len(source_counts),
}

summary_df = pd.DataFrame([summary]).T
summary_df.columns = ["value"]

print("=== DATASET SUMMARY ===")
display(summary_df)

# CLASS IMBALANCE TABLES
# Ensure all classes appear (0..NUM_CLASSES-1) even if missing
img_level = pd.Series({c: image_class_presence.get(c, 0) for c in range(NUM_CLASSES)}, name="images_containing_class")
box_level = pd.Series({c: box_class_counts.get(c, 0) for c in range(NUM_CLASSES)}, name="boxes_of_class")

class_table = pd.concat([img_level, box_level], axis=1)
class_table["img_level_pct"] = class_table["images_containing_class"] / n_images
class_table["box_level_pct"] = class_table["boxes_of_class"] / max(1, int(box_level.sum()))
class_table.index.name = "class_id"

print("\n=== CLASS IMBALANCE (sorted by image-level frequency) ===")
display(class_table.sort_values("images_containing_class", ascending=False))

# Imbalance indicators
nonzero_img = class_table[class_table["images_containing_class"] > 0]["images_containing_class"]
nonzero_box = class_table[class_table["boxes_of_class"] > 0]["boxes_of_class"]

def ratio_max_min(series):
    if len(series) == 0:
        return np.nan
    return float(series.max() / series.min()) if series.min() > 0 else np.inf

imbalance_stats = pd.DataFrame([{
    "img_level_max/min_nonzero": ratio_max_min(nonzero_img),
    "box_level_max/min_nonzero": ratio_max_min(nonzero_box),
    "num_classes_present_img_level": int((class_table["images_containing_class"] > 0).sum()),
    "num_classes_present_box_level": int((class_table["boxes_of_class"] > 0).sum()),
}])

print("\n=== IMBALANCE INDICATORS ===")
display(imbalance_stats)

# MULTI-OBJECT DISTRIBUTION
obj_hist = df["num_objects"].value_counts().sort_index()
obj_dist = pd.DataFrame({
    "num_images": obj_hist,
    "pct": obj_hist / n_images
})
obj_dist.index.name = "num_objects_in_image"

print("\n=== OBJECT COUNT PER IMAGE ===")
display(obj_dist)

# BOX AREA DISTRIBUTION
if all_areas:
    areas = np.array(all_areas, dtype=float)
    # Quantiles give a nice quick view
    q = np.quantile(areas, [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0])
    area_summary = pd.DataFrame({
        "quantile": ["0%", "10%", "25%", "50%", "75%", "90%", "100%"],
        "w*h (normalized)": q
    })
    print("\n=== BOX AREA (w*h) QUANTILES ===")
    display(area_summary)

    # Bucket distribution
    buckets = pd.cut(
        areas,
        bins=[-1e-9, SMALL_AREA_TH, LARGE_AREA_TH, 1.0],
        labels=[f"small(<{SMALL_AREA_TH})", f"medium([{SMALL_AREA_TH},{LARGE_AREA_TH}])", f"large(>{LARGE_AREA_TH})"]
    )
    bucket_counts = buckets.value_counts().sort_index()
    bucket_df = pd.DataFrame({
        "num_boxes": bucket_counts,
        "pct": bucket_counts / len(areas)
    })
    print("\n=== BOX SIZE BUCKETS (by area) ===")
    display(bucket_df)
else:
    print("\nNo boxes found in any txt files (all empty).")


# Save outputs for report
OUT_DIR = Path("analysis_cache")
OUT_DIR.mkdir(exist_ok=True)

summary_df.to_csv(OUT_DIR / "dataset_summary.csv")
class_table.to_csv(OUT_DIR / "class_imbalance.csv")
obj_dist.to_csv(OUT_DIR / "objects_per_image.csv")
df.to_csv(OUT_DIR / "per_image_metadata.csv", index=False)

print(f"\nSaved CSVs to: {OUT_DIR.resolve()}")

=== DATASET SUMMARY ===


,value
images_total,9525.000000
images_with_objects,9525.000000
images_empty,0.000000
pct_empty,0.000000
images_multi_object(>=2),4396.000000
pct_multi_object_over_all,0.461522
pct_multi_object_over_nonempty,0.461522
images_with_small_object(area<0.01),2667.000000
pct_small_object_images,0.280000
images_with_large_object(area>0.1),5860.000000



=== CLASS IMBALANCE (sorted by image-level frequency) ===


,images_containing_class,boxes_of_class,img_level_pct,box_level_pct
class_id,,,,
10,1933,6368,0.202940,0.173416
3,1133,2793,0.118950,0.076060
1,1000,1379,0.104987,0.037553
14,975,1495,0.102362,0.040712
4,949,1103,0.099633,0.030037
16,866,1193,0.090919,0.032488
17,706,782,0.074121,0.021296
18,695,765,0.072966,0.020833
19,689,9330,0.072336,0.254078



=== IMBALANCE INDICATORS ===


...


=== BOX AREA (w*h) QUANTILES ===


,quantile,w*h (normalized)
0,0%,0.000005
1,10%,0.000967
2,25%,0.003711
3,50%,0.019453
4,75%,0.078625
5,90%,0.271875
6,100%,1.000000



=== BOX SIZE BUCKETS (by area) ===


,num_boxes,pct
small(<0.01),14427,0.392881
"medium([0.01,0.1])",14552,0.396286
large(>0.1),7742,0.210833



Saved CSVs to: C:\Users\Usuario\Downloads\techtrack-hortner87-main(4)\techtrack-hortner87-main\analysis\analysis_cache


There is a substantial fraction of multi-object scenes: 4396 images (46.15%) contain at least two objects, and object density can be very high in a few cases (up to 224 objects in a single image). Object scale is highly varied: 28% of images include at least one very small object (area < 0.01), while 61.5% include at least one large object (area > 0.1). At the box level, small and medium objects dominate (about 39.3% small and 39.6% medium), but there is still a meaningful tail of large objects (~21.1%), indicating the model must handle both fine-detail detection and large-instance localization.

Class distribution is clearly imbalanced. The most frequent class appears in ~20.3% of images, while the rarest classes appear in ~1.9–2.4% of images, giving an image-level max/min ratio of ~10.6×; the imbalance is even stronger at the box level (max/min ~36.4×), suggesting some classes contribute many more boxes per image than others. Overall, the dataset combines class imbalance with wide variation in object counts and object scales, which typically motivates sampling strategies or loss reweighting to avoid overfitting to frequent, dense classes and to maintain performance on small objects.

## Configuration

In [5]:
REPO_ROOT = Path.cwd().resolve().parent

# Dataset location (try both common layouts)
DATASET_DIR = REPO_ROOT / "techtrack" / "storage" / "logistics"
assert DATASET_DIR.exists(), f"Dataset dir not found: {DATASET_DIR.resolve()}"

# Number of classes (read from logistics.names if present)
NAMES_PATH = REPO_ROOT / "techtrack" / "storage" / "yolo_models" / "logistics.names"

if NAMES_PATH.exists():
    CLASS_NAMES = [ln.strip() for ln in NAMES_PATH.read_text().splitlines() if ln.strip()]
    NUM_CLASSES = len(CLASS_NAMES)
else:
    CLASS_NAMES = [f"class_{i}" for i in range(20)]
    NUM_CLASSES = 20

print("DATASET_DIR:", DATASET_DIR)
print("NAMES_PATH :", NAMES_PATH if NAMES_PATH.exists() else "(not found)")
print("NUM_CLASSES:", NUM_CLASSES)
print("First classes:", CLASS_NAMES[:10])

DATASET_DIR: C:\Users\Usuario\Downloads\techtrack-hortner87-main(4)\techtrack-hortner87-main\techtrack\storage\logistics
NAMES_PATH : C:\Users\Usuario\Downloads\techtrack-hortner87-main(4)\techtrack-hortner87-main\techtrack\storage\yolo_models\logistics.names
NUM_CLASSES: 20
First classes: ['barcode', 'car', 'cardboard box', 'fire', 'forklift', 'freight container', 'gloves', 'helmet', 'ladder', 'license plate']


In [8]:
# Small/large object thresholds (normalized YOLO area = w*h)
SMALL_AREA_TH = 0.01
LARGE_AREA_TH = 0.10

# Sampling hyperparameters
alpha = 0.5   # rarity strength: w_class = 1 / freq^alpha
beta  = 1.5   # small-object boost multiplier

# Sampling outputs
OUT_DIR = Path("analysis_cache_sampling")
OUT_DIR.mkdir(exist_ok=True)

# How many samples to draw
SAMPLE_K_NO_REPLACEMENT = 5000

# Copy sampled files into a folder (can be large; set False if not needed)
COPY_FILES = True
SAMPLED_COPY_DIR = OUT_DIR / "sampled_subset"


DATASET_DIR: storage\logistics
NAMES_PATH : storage\yolo_models\logistics.names
NUM_CLASSES: 20
First classes: ['barcode', 'car', 'cardboard box', 'fire', 'forklift', 'freight container', 'gloves', 'helmet', 'ladder', 'license plate']


## 2) Parse YOLO annotations and scan dataset

In [9]:

def parse_yolo_txt(txt_path: Path):
    """Returns list of (class_id, x, y, w, h) from YOLO txt."""
    if not txt_path.exists():
        return []
    text = txt_path.read_text().strip()
    if not text:
        return []
    rows = []
    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            rows.append((c, x, y, w, h))
        except ValueError:
            continue
    return rows

# List images
image_paths = sorted(list(DATASET_DIR.glob("*.jpg")) + list(DATASET_DIR.glob("*.png")))
assert image_paths, f"No images found in {DATASET_DIR.resolve()}"

records = []
box_class_counts = Counter()
image_class_presence = Counter()

for img_path in tqdm(image_paths, desc="Scanning dataset"):
    txt_path = img_path.with_suffix(".txt")
    ann = parse_yolo_txt(txt_path)

    classes_in_img = {c for c, *_ in ann}
    num_objects = len(ann)

    # Size bucket flags
    areas = [max(0.0, w) * max(0.0, h) for (c, x, y, w, h) in ann]
    has_small = any(a < SMALL_AREA_TH for a in areas) if areas else False
    has_large = any(a > LARGE_AREA_TH for a in areas) if areas else False

    # Update class counts
    for c in classes_in_img:
        image_class_presence[int(c)] += 1
    for c, *_ in ann:
        box_class_counts[int(c)] += 1

    records.append({
        "image": str(img_path),
        "annotation": str(txt_path),
        "num_objects": int(num_objects),
        "classes_sorted": sorted(list(map(int, classes_in_img))),
        "num_unique_classes": int(len(classes_in_img)),
        "has_small_object": bool(has_small),
        "has_large_object": bool(has_large),
    })

df = pd.DataFrame(records)
n_images = len(df)
print("Images scanned:", n_images)

# Save raw scan
df.to_csv(OUT_DIR / "per_image_metadata.csv", index=False)

# Build class frequency table (image-level)
img_freq = {c: int(image_class_presence.get(c, 0)) for c in range(NUM_CLASSES)}
class_table = pd.DataFrame({
    "class_id": list(range(NUM_CLASSES)),
    "class_name": CLASS_NAMES[:NUM_CLASSES],
    "images_containing_class": [img_freq[c] for c in range(NUM_CLASSES)],
    "boxes_of_class": [int(box_class_counts.get(c, 0)) for c in range(NUM_CLASSES)],
})
class_table["img_level_pct"] = class_table["images_containing_class"] / n_images
class_table["box_level_pct"] = class_table["boxes_of_class"] / max(1, class_table["boxes_of_class"].sum())

display(class_table.sort_values("images_containing_class", ascending=False).head(10))
class_table.to_csv(OUT_DIR / "class_frequency_table.csv", index=False)


Scanning dataset: 100%|██████████████████████████████████████████████████████████| 9525/9525 [00:06<00:00, 1487.21it/s]


Images scanned: 9525


,class_id,class_name,images_containing_class,boxes_of_class,img_level_pct,box_level_pct
10,10,person,1933,6368,0.202940,0.173416
3,3,fire,1133,2793,0.118950,0.076060
1,1,car,1000,1379,0.104987,0.037553
14,14,smoke,975,1495,0.102362,0.040712
4,4,forklift,949,1103,0.099633,0.030037
16,16,traffic light,866,1193,0.090919,0.032488
17,17,truck,706,782,0.074121,0.021296
18,18,van,695,765,0.072966,0.020833
19,19,wood pallet,689,9330,0.072336,0.254078
7,7,helmet,631,2170,0.066247,0.059094


## 3) Primary class per image (rarest class present)
For multi-label images, define the primary class as the class with smallest image-level frequency.

In [10]:

def primary_class(classes_in_img):
    # If an image had no classes, return -1
    if not classes_in_img:
        return -1
    return min(classes_in_img, key=lambda c: img_freq.get(int(c), 1) if img_freq.get(int(c), 0) > 0 else 1)

df["primary_class"] = df["classes_sorted"].apply(primary_class)
df["primary_class_name"] = df["primary_class"].apply(lambda c: CLASS_NAMES[c] if 0 <= c < len(CLASS_NAMES) else "NA")

display(df[["image","num_objects","classes_sorted","primary_class","primary_class_name","has_small_object"]].head(5))


,image,num_objects,classes_sorted,primary_class,primary_class_name,has_small_object
0,storage\logistics\-01-15-1-1-1-2-26_jpg.rf.eb6...,11,"[2, 6, 10]",6,gloves,True
1,storage\logistics\-01-15-1-4-1-1-14_jpg.rf.2e8...,3,"[2, 10]",2,cardboard box,True
2,storage\logistics\-01-15-1-4-1-4-111_jpg.rf.ea...,4,"[2, 10]",2,cardboard box,False
3,storage\logistics\-01-15-1-4-1-5-45_jpg.rf.079...,11,"[2, 6, 10]",6,gloves,True
4,storage\logistics\-01-15-2-2-2-1-212_jpg.rf.c3...,16,"[2, 10]",2,cardboard box,True


## 4) Compute sampling weights

We use:
- **Rare-class weighting**: $w_{class} = 1 / f(c)^{\alpha}$
- **Small-object boost**: multiply by $\beta$ if `has_small_object`
- **Density penalty**: $w_{density} = 1 / \sqrt{1 + n_{objects}}$

Final weight: $w = w_{class} \times w_{small} \times w_{density}$.

In [11]:

def compute_weight(row):
    c = int(row["primary_class"])
    f = float(img_freq.get(c, 1))  # avoid div by zero
    w_class = 1.0 / (f ** float(alpha))

    w_small = float(beta) if bool(row["has_small_object"]) else 1.0
    w_dense = 1.0 / np.sqrt(1.0 + float(row["num_objects"]))

    return w_class * w_small * w_dense

df["sample_weight"] = df.apply(compute_weight, axis=1)

# Convert to probabilities
w = df["sample_weight"].to_numpy(dtype=float)
df["sample_prob"] = w / w.sum()

print("Weight summary:")
display(df["sample_weight"].describe(percentiles=[0.5, 0.75, 0.9, 0.99]))


Weight summary:


count    9525.000000
mean        0.025855
std         0.010175
min         0.003810
50%         0.022954
75%         0.027735
90%         0.041451
99%         0.055442
max         0.078406
Name: sample_weight, dtype: float64

## 5) Validate sampler behavior
We check if the sampler increases representation of rare primary classes and small-object images, by comparing the baseline image-level distribution of primary_class to the distribution obtained after sampling with probabilities ``sample_prob``. The **delta_pct column** shows which classes are upweighted (positive) or downweighted (negative), indicating whether rare primary classes receive more sampling mass than in the raw dataset. In parallel, we compare the baseline versus sampled mean of ``has_small_object`` to verify that images containing small objects are selected more often under the sampler.

In [12]:

# Baseline distributions
base_primary = df["primary_class"].value_counts().sort_index()
base_small_pct = df["has_small_object"].mean()

# Draw a sample with replacement
rng = np.random.default_rng(0)
sampled_idx = rng.choice(df.index.to_numpy(), size=SAMPLE_K_NO_REPLACEMENT, replace=False, p=df["sample_prob"].to_numpy())
sdf = df.loc[sampled_idx].copy()

samp_primary = sdf["primary_class"].value_counts().sort_index()
samp_small_pct = sdf["has_small_object"].mean()

comp = pd.DataFrame({
    "base_count": base_primary,
    "sampled_count": samp_primary
}).fillna(0).astype(int)

comp["base_pct"] = comp["base_count"] / comp["base_count"].sum()
comp["sampled_pct"] = comp["sampled_count"] / comp["sampled_count"].sum()
comp["delta_pct"] = comp["sampled_pct"] - comp["base_pct"]
comp["class_name"] = [CLASS_NAMES[i] for i in comp.index]

print("Base small-object pct:", float(base_small_pct))
print("Sampled small-object pct:", float(samp_small_pct))

display(comp.sort_values("delta_pct", ascending=False).head(10))
display(comp.sort_values("delta_pct", ascending=True).head(10))

comp.to_csv(OUT_DIR / "primary_class_distribution_comparison.csv", index=True)


Base small-object pct: 0.28
Sampled small-object pct: 0.2614


,base_count,sampled_count,base_pct,sampled_pct,delta_pct,class_name
primary_class,,,,,,
5,185,212,0.019423,0.0424,0.022977,freight container
9,268,252,0.028136,0.0504,0.022264,license plate
0,272,235,0.028556,0.0470,0.018444,barcode
6,219,202,0.022992,0.0404,0.017408,gloves
11,284,227,0.029816,0.0454,0.015584,qr code
8,183,168,0.019213,0.0336,0.014387,ladder
15,286,207,0.030026,0.0414,0.011374,traffic cone
12,299,199,0.031391,0.0398,0.008409,road sign
18,628,343,0.065932,0.0686,0.002668,van


,base_count,sampled_count,base_pct,sampled_pct,delta_pct,class_name
primary_class,,,,,,
4,900,376,0.094488,0.0752,-0.019288,forklift
3,767,307,0.080525,0.0614,-0.019125,fire
10,399,114,0.041890,0.0228,-0.019090,person
14,975,418,0.102362,0.0836,-0.018762,smoke
19,653,270,0.068556,0.0540,-0.014556,wood pallet
1,745,319,0.078215,0.0638,-0.014415,car
2,376,148,0.039475,0.0296,-0.009875,cardboard box
13,445,190,0.046719,0.0380,-0.008719,safety vest
7,209,81,0.021942,0.0162,-0.005742,helmet


The comparison shows the sampler is behaving as intended: it shifts sampling probability toward several low-frequency primary classes and away from more common ones. In the "most increased" list, classes like **freight container**, **license plate**, **barcode**, **gloves**, and **QR code** roughly double their image-level share in the sampled set. In contrast, the "most decreased" list is dominated by higher base frequency classes such as **forklift**, **fire**, **person**, **smoke**, and **wood pallet**, whose sampled shares drop by around 1–2 percentage points. Overall, the sampled distribution is more balanced, indicating the sampler increases exposure to rarer primary classes while reducing over-representation of frequent ones.

## 6) Produce sampled subsets

We sample **without replacement**: this is appropriate for creating a balanced subset for evaluation, which is our purpose.

In [13]:
# Without replacement sample subset (size = SAMPLE_K_NO_REPLACEMENT)
sampled_no_repl = df.sample(
    n=min(SAMPLE_K_NO_REPLACEMENT, len(df)),
    replace=False,
    weights="sample_weight",
    random_state=0
)[["image","annotation","primary_class","has_small_object","num_objects","sample_weight"]].reset_index(drop=True)
sampled_no_repl.to_csv(OUT_DIR / "sample_without_replacement.csv", index=False)
print("Saved:", OUT_DIR / "sample_without_replacement.csv")

sampled_no_repl.head(5)


Saved: analysis_cache_sampling\sample_without_replacement.csv


,image,annotation,primary_class,has_small_object,num_objects,sample_weight
0,storage\logistics\IMAG0532_png_jpg.rf.ee46e420...,storage\logistics\IMAG0532_png_jpg.rf.ee46e420...,0,False,1,0.042875
1,storage\logistics\OIF-42-_jpg.rf.53378c0e0bfe6...,storage\logistics\OIF-42-_jpg.rf.53378c0e0bfe6...,1,False,1,0.022361
2,storage\logistics\image_from_china-743-_jpg.rf...,storage\logistics\image_from_china-743-_jpg.rf...,7,True,8,0.019905
3,storage\logistics\IMAG0396_jpg.rf.28fd9584e4d9...,storage\logistics\IMAG0396_jpg.rf.28fd9584e4d9...,2,False,1,0.033596
4,storage\logistics\fire11_mp4-18_jpg.rf.673ca05...,storage\logistics\fire11_mp4-18_jpg.rf.673ca05...,14,True,4,0.021483


## 7) Copy sampled images/labels to a folder

This creates a new mini-dataset folder with the sampled images and `.txt` files.

In [17]:
COPY_FILES = True

if COPY_FILES:
    SAMPLED_COPY_DIR.mkdir(parents=True, exist_ok=True)

    for _, row in tqdm(sampled_no_repl.iterrows(), total=len(sampled_no_repl), desc="Copying files"):
        img_src = Path(row["image"])
        lbl_src = Path(row["annotation"])

        # Copy image into the same folder
        shutil.copy2(img_src, SAMPLED_COPY_DIR / img_src.name)

        # Copy label into the same folder (same stem, .txt extension)
        if lbl_src.exists():
            shutil.copy2(lbl_src, SAMPLED_COPY_DIR / lbl_src.name)

    # Also copy names file if available (stays in the same folder too)
    if NAMES_PATH.exists():
        shutil.copy2(NAMES_PATH, SAMPLED_COPY_DIR / NAMES_PATH.name)

    print("Copied subset to:", SAMPLED_COPY_DIR.resolve())
else:
    print("COPY_FILES=False (skipping copy step)")

Copying files: 100%|██████████████████████████████████████████████████████████████| 5000/5000 [00:34<00:00, 145.78it/s]

Copied subset to: C:\Users\Usuario\Downloads\techtrack-hortner87-main(4)\techtrack-hortner87-main\techtrack\analysis_cache_sampling\sampled_subset
